# Exp 27: Smaller image sizes

Card: [exp27_small_images.md](../../docs/experiments/exp27_small_images.md)


In [ ]:
import matplotlib.pyplot as plt

from core.training.config import ROOT
from core.training.cv import CVResult
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import pr_curves, save_figures, use_style

N_SAMPLE = 16  # exp19 sample
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
CONFIGS = {
    f"{size} px": CONFIG_DIR / "exp27_small_images" / f"resnet18_{size}.yaml"
    for size in (128, 64)
}
EXP20 = CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp27_small_images"

# dataviz reference palette, light mode
SERIES = {"224 px": "#2a78d6", "128 px": "#1baf7a", "64 px": "#eb6834"}
use_style()

## Training


In [ ]:
results = {"224 px": CVResult.load(EXP20)}
results.update({name: CVResult.load(cfg) for name, cfg in CONFIGS.items()})
figures = {}

## Grad-CAM Cat, Dog: 224 vs 128 vs 64 px


In [ ]:
cams = {"224 px": GradCamFigures(EXP20)} | {
    name: GradCamFigures(cfg) for name, cfg in CONFIGS.items()
}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = exp16.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)